# Prepare dataset with new features for ML model training #

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, isnull, when
import pandas as pd

# Initialize Spark session
spark = SparkSession.builder \
    .appName("ML Features") \
    .getOrCreate()

print('spark initialised')

24/06/30 14:21:50 WARN Utils: Your hostname, goumbp.local resolves to a loopback address: 127.0.0.1; using 192.168.1.136 instead (on interface en0)
24/06/30 14:21:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/06/30 14:21:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark initialised


## Load data sets ##

In [2]:
'''
 We load the data.csv of the joined data set of features of SQL assignment 1
'''

# File path to the input CSV file
csv_file_path = "/Users/tgou1055/jr_data/jr_project/imba_data/output"

# Read the CSV file with schema inference
data_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_file_path)

# File path to the input CSV file
csv_file_path = "/Users/tgou1055/jr_data/jr_project/imba_data/order_products/order_products__train.csv"

# Read the CSV file with schema inference
order_products_train_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_file_path)

# File path to the input CSV file
csv_file_path = "/Users/tgou1055/jr_data/jr_project/imba_data/orders/orders.csv"

# Read the CSV file with schema inference
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_file_path)

print("load data complete")

load data complete


## Prepare training set with feature dataset

In [3]:
# rename column order_id to order_id_1
filter_orders_df = orders_df[['user_id','order_id']].withColumnRenamed('order_id', 'order_id_1')

# add order_id to order_products_train
order_products_train_df = order_products_train_df.join(filter_orders_df, \
                                                       order_products_train_df['order_id'] == filter_orders_df['order_id_1'], 'inner')
order_products_train_new_df = order_products_train_df.drop('order_id_1')
#order_products_train_new_df.show(2)

print('transformation complete')

transformation complete


In [3]:
#orders_df.filter( (col('eval_set')== 'test') & (col('user_id')== 3) ).show(5)

In [4]:
# attach eval_set to data
'''
In the following transformation, we add 'eval_set' infomation to the features 'data' we imported.
Notice that for each 'user_id' with 'eval_set = train or test', there is only one 'order_id'
( This is reasonable, because we try to train the model with the lastest orders of users,
  with the features deduced with 'eval_set = prior'
)
'''
filtered_orders_2_df = orders_df[orders_df['eval_set'] != 'prior'][['user_id','eval_set']].withColumnRenamed('user_id', 'user_id_1')
data_df = data_df.join(filtered_orders_2_df, data_df['user_id']==filtered_orders_2_df['user_id_1'], 'inner').drop('user_id_1')

In [6]:
#order_products_train_new_df.filter(col('reordered') == 1).count()

In [5]:
# attach target variable: reordered
'''
The data now contains all the prepared features for those orders of a user that has 
    eval_set = 'train' or 'test' 

    We use the left join to join 'data' and order_products_train
'''
order_products_train_new_df = order_products_train_new_df[['user_id', 'product_id', 'reordered']]
order_products_train_new_df = order_products_train_new_df.withColumnRenamed('user_id','user_id_1').withColumnRenamed('product_id','product_id_1')

data_new_df = data_df.join(order_products_train_new_df, \
        (data_df['user_id'] == order_products_train_new_df['user_id_1'])&(data_df['product_id'] == order_products_train_new_df['product_id_1']), \
        'left').drop('user_id_1').drop('product_id_1')

print('transformation complete')

transformation complete


In [6]:
#data_new_df.printSchema()
data_new_df[['user_id','product_id','eval_set','reordered']].filter( col('reordered') == 0 ).show()

+-------+----------+--------+---------+
|user_id|product_id|eval_set|reordered|
+-------+----------+--------+---------+
+-------+----------+--------+---------+



## Implement more features to data

In [7]:
data_new_df = data_new_df.withColumn('prod_reorder_probability', col('seq_is_two')/col('seq_is_one'))
data_new_df = data_new_df.withColumn('prod_reorder_times', 1 + col('sum_reordered')/col('seq_is_one'))
data_new_df = data_new_df.withColumn('prod_reorder_ratio', col('sum_reordered')/col('num_product_ordered'))
data_new_df = data_new_df.drop('sum_reordered', 'seq_is_one', 'seq_is_two')
data_new_df = data_new_df.withColumn('user_average_basket', col('num_product_bought')/col('max_order_number'))
data_new_df = data_new_df.withColumn('up_order_rate', col('number_of_orders')/col('max_order_number'))
data_new_df = data_new_df.withColumn('up_orders_since_last_order', col('max_order_number') - col('max_order_num'))
data_new_df = data_new_df.withColumn('up_order_rate_since_first_order', col('number_of_orders') / (col('max_order_number') - col('min_order_num')+1))

In [9]:
# data_new_df[['user_id','product_id','prod_reorder_probability',\
#          'prod_reorder_times','prod_reorder_ratio','user_average_basket', \
#          'up_order_rate','up_orders_since_last_order','up_order_rate_since_first_order']].show(5)

In [8]:
# store the data in csv file
data_new_df = data_new_df.coalesce(1)
output_path = "/Users/tgou1055/jr_data/jr_project/imba_data/output2/"
data_new_df.write.mode('overwrite').csv(output_path, header=True)

## Prepare train set and test set

In [9]:
# split into training and test set, test set does not have target variable
train_df = data_new_df.filter(data_new_df['eval_set'] == 'train')
test_df = data_new_df.filter(data_new_df['eval_set'] == 'test')

In [10]:
# id field won't be used in model, thus make a backup of them and remove from dataframe
test_id_df = test_df[['product_id','user_id', 'eval_set']]
test_df = test_df.drop('product_id','user_id', 'eval_set', 'reordered')

In [11]:
# convert target variable to 1/0 for training dataframe
from pyspark.sql.types import IntegerType, StringType
#train_df['reordered'] = train_df['reordered'].fillna(0)
train_df = train_df.fillna({'reordered': 0})
train_df = train_df.withColumn('reordered', train_df['reordered'].cast(IntegerType()))

In [14]:
#train_df.show()

In [13]:
train_df.filter((col('reordered') != 1) & (col('reordered') != 0))['user_id','product_id','reordered'].show(10)

+-------+----------+---------+
|user_id|product_id|reordered|
+-------+----------+---------+
+-------+----------+---------+



In [12]:
# drop id columns as they won't be used in model
train_df = train_df.drop('eval_set', 'user_id', 'product_id')
df = train_df

In [13]:
# Split the DataFrame into training (80%) and validation (20%) sets
train_df, validation_df = df.randomSplit([0.8, 0.2], seed=42)

In [18]:
# This is the target variable dataframe
train_y = train_df[['reordered']]
validation_y = validation_df[['reordered']]
# This is the dataframe without target variable and contains all the features
train_X = train_df.drop('reordered')
validation_X = validation_df.drop('reordered')

In [14]:
# Reorder columns
columns = ['reordered'] + [col for col in df.columns if col != 'reordered']
train_new_df = train_df.select(columns)

In [15]:
# Reorder columns
columns = ['reordered'] + [col for col in df.columns if col != 'reordered']
validation_new_df = validation_df.select(columns)

In [16]:
train_new_df.printSchema()
train_new_df.count()

root
 |-- reordered: integer (nullable = false)
 |-- max_order_number: integer (nullable = true)
 |-- max_order_num: integer (nullable = true)
 |-- num_product_bought: integer (nullable = true)
 |-- min_order_num: integer (nullable = true)
 |-- avg_days_prior: double (nullable = true)
 |-- sum_days_prior: integer (nullable = true)
 |-- num_distinct_product_bought: integer (nullable = true)
 |-- seq_add_to_order: double (nullable = true)
 |-- reordered_ratio: double (nullable = true)
 |-- number_of_orders: integer (nullable = true)
 |-- num_product_ordered: integer (nullable = true)
 |-- prod_reorder_probability: double (nullable = true)
 |-- prod_reorder_times: double (nullable = true)
 |-- prod_reorder_ratio: double (nullable = true)
 |-- user_average_basket: double (nullable = true)
 |-- up_order_rate: double (nullable = true)
 |-- up_orders_since_last_order: integer (nullable = true)
 |-- up_order_rate_since_first_order: double (nullable = true)



6780073

In [17]:
validation_new_df.printSchema()
validation_new_df.count()

root
 |-- reordered: integer (nullable = false)
 |-- max_order_number: integer (nullable = true)
 |-- max_order_num: integer (nullable = true)
 |-- num_product_bought: integer (nullable = true)
 |-- min_order_num: integer (nullable = true)
 |-- avg_days_prior: double (nullable = true)
 |-- sum_days_prior: integer (nullable = true)
 |-- num_distinct_product_bought: integer (nullable = true)
 |-- seq_add_to_order: double (nullable = true)
 |-- reordered_ratio: double (nullable = true)
 |-- number_of_orders: integer (nullable = true)
 |-- num_product_ordered: integer (nullable = true)
 |-- prod_reorder_probability: double (nullable = true)
 |-- prod_reorder_times: double (nullable = true)
 |-- prod_reorder_ratio: double (nullable = true)
 |-- user_average_basket: double (nullable = true)
 |-- up_order_rate: double (nullable = true)
 |-- up_orders_since_last_order: integer (nullable = true)
 |-- up_order_rate_since_first_order: double (nullable = true)



1694588

In [18]:
train = train_new_df.coalesce(1)
validation= validation_new_df.coalesce(1)

path_train = "/Users/tgou1055/jr_data/jr_project/imba_data/output3/train/"
path_validation = "/Users/tgou1055/jr_data/jr_project/imba_data/output3/validation/"
train.write.mode('overwrite').csv(path_train, header=False)
validation.write.mode('overwrite').csv(path_validation, header=False)

print('file saving complete')

file saving complete


In [19]:
test_id = test_id_df.coalesce(1)
test = test_df.coalesce(1)
path_test_id = "/Users/tgou1055/jr_data/jr_project/imba_data/output3/test_id/"
path_test = "/Users/tgou1055/jr_data/jr_project/imba_data/output3/test/"
test_id.write.mode('overwrite').csv(path_test_id, header=False)
test.write.mode('overwrite').csv(path_test, header=False)

print('file saving complete')

file saving complete


In [20]:
spark.stop()